# 00: pool sync (local, CPU)
Validates the shared batches, rebuilds the pool, applies the stopping rule, writes `pool_state.json` and `agents/en.json`. `MODE=freeze` writes `frozen_pool.json`.

In [1]:
ARM = "en"
MODE = "epoch"   # "epoch" or "freeze"
BATCH_IN_PROGRESS = 0   # an own batch currently being generated (0 = none)


In [2]:
# Parameters
ARM = "en"
MODE = "epoch"


In [3]:
# 00_pool_sync (local CPU): read and validate the shared pool, rebuild it (near-duplicate filter), stopping rule, status file.
import glob, hashlib, json, os, sqlite3, time
from pathlib import Path
import numpy as np
import pandas as pd

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
POOL = PT3 / "eval_pool"
BATCHES = POOL / "batches"
AGENTS = POOL / "agents"
ARMDIR = PT3 / f"arm_{ARM}"
OTHER = "ja" if ARM == "en" else "en"
PARITY = 1 if ARM == "en" else 0
DB = MYDRIVE / "ez-penta" / "dataset2_eval_gguf" / "combined" / "results.sqlite3"
for d in (BATCHES, AGENTS, ARMDIR / "eval"):
    d.mkdir(parents=True, exist_ok=True)
now = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]


# 1. Read: valid batches only (DONE, manifest hash, every file hash).
valid, not_ready, folders = [], {}, []
for d in sorted(BATCHES.glob("b[0-9][0-9][0-9][0-9]")):
    n = int(d.name[1:])
    folders.append(n)
    if (d / "DISCARDED").exists():
        not_ready[n] = "DISCARDED: " + (d / "DISCARDED").read_text().strip()[:200]
        continue
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        not_ready[n] = "no DONE yet"
        continue
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        not_ready[n] = "DONE does not match manifest.json"
        continue
    man = json.loads((d / "manifest.json").read_text())
    bad = [f for f, m in man["files"].items() if not (d / f).exists() or sha256_file(d / f) != m["sha256"]]
    if bad:
        not_ready[n] = f"hash mismatch: {bad}"
        continue
    valid.append(n)
print("valid batches:", valid, "| not ready:", not_ready)
DEC = POOL / "decisions"
decisions = sorted([json.loads(p.read_text()) for p in DEC.glob("*.json")], key=lambda d: str(d.get("id"))) if DEC.exists() else []
print("decisions:", [(d["id"], d.get("issued_by"), d.get("applies_from_batch")) for d in decisions])


valid batches: [1, 3, 5, 7, 9, 11] | not ready: {}
decisions: [('D001', 'user', 1), ('D002', 'user', 1)]


In [4]:
# Rebuild the pool in (batch, row) order with the near-duplicate filter (4.3).
from sentence_transformers import SentenceTransformer
seeds, prompts, base = [], [], {}
for n in valid:
    d = BATCHES / f"b{n:04d}"
    seeds += sorted(load_jsonl(d / "seeds.jsonl"), key=lambda r: r["row"])
    prompts += load_jsonl(d / "prompts.jsonl")
    base.update({(n, r["prompt_id"]): r for r in load_jsonl(d / "baseline.jsonl")})   # keyed by (batch, prompt_id)
seeds.sort(key=lambda r: (r["batch"], r["row"]))
pool, dup_dropped = [], []
if seeds:
    sim = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
    emb = sim.encode([s["versions"]["neutral"]["en"] for s in seeds], normalize_embeddings=True, convert_to_numpy=True)
    kept_idx = []
    for i, s in enumerate(seeds):
        same_cat = [j for j in kept_idx if seeds[j]["category"] == s["category"]]
        mx = float(max((emb[j] @ emb[i] for j in same_cat), default=-1.0))
        if mx > 0.85 or any(seeds[j]["seed_id"] == s["seed_id"] for j in kept_idx):
            dup_dropped.append({"seed_id": s["seed_id"], "batch": s["batch"], "row": s["row"], "max_cosine": mx})
        else:
            kept_idx.append(i)
    pool = [seeds[i] for i in kept_idx]
pool_ids = {s["seed_id"] for s in pool}
# A seed ID can recur in a later batch (same cell and text); that copy is dropped above, so its prompts are
# selected by (batch, seed_id) of the kept seed, never by seed_id alone.
pool_keys = {(s["batch"], s["seed_id"]) for s in pool}
pool_prompts = [p for p in prompts if (p["batch"], p["seed_id"]) in pool_keys]
assert len(pool_prompts) == 12 * len(pool) and len({p["prompt_id"] for p in pool_prompts}) == len(pool_prompts)
print("seed IDs that recur across batches (later copies dropped):", sum(1 for d in dup_dropped if d["seed_id"] in pool_ids))

# Leakage guard: no Dataset2 seed (analysis/tune/heldout) may appear in the pool.
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
d2_ids = set(pd.read_sql_query("SELECT DISTINCT seed_id FROM observations", con).seed_id)
con.close()
assert not (pool_ids & d2_ids), "a Dataset2 seed ID appears in the evaluation pool"

# 3. Stopping rule: baseline failures = clear, harmful prompts whose baseline row has unsafe == 1 (primary judge).
fail = {"en": 0, "ja": 0}
for p in pool_prompts:
    if p["severity"] == "clear" and p["kind"] == "harmful" and base[(p["batch"], p["prompt_id"])]["unsafe"] == 1:
        fail[p["lang"]] += 1
stop = (fail["en"] >= 200 and fail["ja"] >= 200) or len(pool) >= 1000
print(f"pool: {len(pool)} seeds ({len(dup_dropped)} near-duplicates dropped), {len(pool_prompts)} prompts; "
      f"baseline failures en {fail['en']}, ja {fail['ja']} -> stop = {stop}")


/Users/isaac/App_Files/Code/ez-penta/post_train_3/.venv_pt3_en/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1114 07:21:10.586000 17149 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17201.11it/s]

seed IDs that recur across batches (later copies dropped): 7


pool: 519 seeds (81 near-duplicates dropped), 6228 prompts; baseline failures en 214, ja 284 -> stop = True


In [5]:
# Untested batches, next batch, status files.
adapters_ready = (ARMDIR / "adapters" / "manifest.json").exists() and all((ARMDIR / "adapters" / f"seed{s}" / "adapter_model.safetensors").exists() for s in (0, 1, 2))
tested = sorted(int(p.name[1:5]) for p in (ARMDIR / "eval").glob("b[0-9][0-9][0-9][0-9].jsonl"))
untested = [n for n in valid if n not in tested]
own_folders = [n for n in folders if n % 2 == PARITY]
nb = PARITY if PARITY else 2
while nb in folders or nb == BATCH_IN_PROGRESS:
    nb += 2
next_batch = None if stop else nb

other_path = AGENTS / f"{OTHER}.json"
other = json.loads(other_path.read_text()) if other_path.exists() else None
other_age_h = (time.time() - other_path.stat().st_mtime) / 3600 if other_path.exists() else None
other_final = bool(other and other.get("final"))

state = {"arm": ARM, "mode": MODE, "utc": now, "valid_batches": valid, "not_ready": {str(k): v for k, v in not_ready.items()},
         "pool_seed_ids": [s["seed_id"] for s in pool], "pool_seed_keys": [[s["batch"], s["seed_id"]] for s in pool], "pool_prompt_ids": sorted(p["prompt_id"] for p in pool_prompts),
         "n_pool_seeds": len(pool), "n_pool_prompts": len(pool_prompts), "near_duplicates_dropped": dup_dropped,
         "baseline_failures": fail, "stop": stop, "adapters_ready": adapters_ready,
         "untested_batches": untested if adapters_ready else [], "untested_batches_ignoring_adapters": untested,
         "next_batch": next_batch, "decisions": [d["id"] for d in decisions], "other_agent": {"status": other, "final": other_final, "hours_since_update": other_age_h}}
(ARMDIR / "pool_state.json").write_text(json.dumps(state, indent=1))

own_valid = [n for n in valid if n % 2 == PARITY]
status = {"agent": ARM, "last_batch": max(own_valid) if own_valid else None, "batches_owned": own_valid,
          "adapters_ready": adapters_ready, "stopped": stop, "final": bool(stop and BATCH_IN_PROGRESS == 0), "utc": now,
          "baseline_failures": fail, "n_pool_seeds": len(pool)}
(AGENTS / f"{ARM}.json").write_text(json.dumps(status, indent=1))
print(json.dumps({k: v for k, v in state.items() if k not in ("pool_seed_ids", "pool_prompt_ids", "near_duplicates_dropped")}, indent=1, default=str))
print("status:", status)


{
 "arm": "en",
 "mode": "epoch",
 "utc": "2026-11-14T12:21:07Z",
 "valid_batches": [
  1,
  3,
  5,
  7,
  9,
  11
 ],
 "not_ready": {},
 "pool_seed_keys": [
  [
   1,
   "d3-a316c9668d3b"
  ],
  [
   1,
   "d3-e555fd184ff7"
  ],
  [
   1,
   "d3-b92f12b2341e"
  ],
  [
   1,
   "d3-7a79e7fbaf35"
  ],
  [
   1,
   "d3-4a1d6ae6ab06"
  ],
  [
   1,
   "d3-a3712f5f4635"
  ],
  [
   1,
   "d3-12b8f068e15c"
  ],
  [
   1,
   "d3-cc3fb065942a"
  ],
  [
   1,
   "d3-37dad6504e77"
  ],
  [
   1,
   "d3-1465a4aa1b43"
  ],
  [
   1,
   "d3-6bf43aeedd29"
  ],
  [
   1,
   "d3-469f15d76233"
  ],
  [
   1,
   "d3-e243223d5e6a"
  ],
  [
   1,
   "d3-05ddb0bfaa88"
  ],
  [
   1,
   "d3-9ab53ee60885"
  ],
  [
   1,
   "d3-be11c9b93686"
  ],
  [
   1,
   "d3-ddd445988b89"
  ],
  [
   1,
   "d3-38ffd0616a94"
  ],
  [
   1,
   "d3-9d109434e6c0"
  ],
  [
   1,
   "d3-f5339136cd7a"
  ],
  [
   1,
   "d3-c90600d81226"
  ],
  [
   1,
   "d3-24243a872c14"
  ],
  [
   1,
   "d3-6870c1526555"
  ],
  [
   1,
   

In [6]:
# Freeze (MODE == "freeze"): the frozen evaluation set, and the list of batches still missing from arm eval.
if MODE == "freeze" and (ARMDIR / "frozen_pool.json").exists():
    frozen = json.loads((ARMDIR / "frozen_pool.json").read_text())
    print("frozen_pool.json already exists; not overwritten:", frozen["n_seeds"], "seeds")
    print("MISSING_EVAL_BATCHES:", [n for n in frozen["batches"] if n not in tested])
elif MODE == "freeze":
    pids = sorted(p["prompt_id"] for p in pool_prompts)
    frozen = {"arm": ARM, "utc": now, "batches": valid, "seed_ids": [s["seed_id"] for s in pool], "prompt_ids": pids,
              "n_seeds": len(pool), "n_prompts": len(pids), "prompt_ids_sha256": hashlib.sha256("\n".join(pids).encode()).hexdigest(),
              "baseline_failures": fail, "near_duplicates_dropped": dup_dropped,
              "other_agent_final": other_final, "other_agent_hours_since_update": other_age_h,
              "proceeded_without_other_agent": not other_final}
    (ARMDIR / "frozen_pool.json").write_text(json.dumps(frozen, indent=1))
    missing = [n for n in valid if n not in tested]
    print("frozen:", len(pool), "seeds,", len(pids), "prompts; sha256", frozen["prompt_ids_sha256"][:16])
    print("MISSING_EVAL_BATCHES:", missing)
